# Notebook 03: ABSA Model Evaluation & Benchmarking
### Project: AspectSense AI
**Author:** Raj Keshav | **Track:** NLP & LLM Capstone 2026

This notebook covers:
1. Aspect-level polarity scoring with hyperbolic tangent normalization.
2. Emotion classification and urgency triage.
3. Confusion matrix, Precision, Recall, and Macro F1 evaluation across the benchmark set.


In [1]:
import sys
import os
import pandas as pd
sys.path.insert(0, os.path.abspath(".."))

from code.core.pipeline import AspectSensePipeline
from code.utils.metrics import calculate_classification_metrics

pipeline = AspectSensePipeline()
df = pd.read_csv("../resources/data/customer_reviews_benchmark.csv")

y_true = []
y_pred = []

for _, row in df.iterrows():
    res = pipeline.analyze_review(row["review_text"], domain=row["domain"], include_llm_diagnostics=False)
    y_true.append(row["overall_sentiment"].strip().lower())
    y_pred.append(res["overall_assessment"]["sentiment"].strip().lower())

metrics = calculate_classification_metrics(y_true, y_pred)
print("Accuracy   :", metrics["accuracy"])
print("Macro F1   :", metrics["macro_f1"])
print("Weighted F1:", metrics["weighted_f1"])


### 3.2 Confusion Matrix Heatmap


In [1]:
import matplotlib.pyplot as plt
import numpy as np

classes = metrics["classes"]
matrix = np.zeros((len(classes), len(classes)))
for i, c1 in enumerate(classes):
    for j, c2 in enumerate(classes):
        matrix[i, j] = metrics["confusion_matrix"][c1].get(c2, 0)

plt.figure(figsize=(6, 5))
plt.imshow(matrix, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(classes)), classes)
plt.yticks(range(len(classes)), classes)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("ABSA Sentiment Confusion Matrix")
plt.show()
